<a href="https://colab.research.google.com/github/Sunitha-k2006/sunitha-codeboosters-internship-2026/blob/main/Phase_01_Data_Engineering/Day_04_BigData_PySpark_Architecture/Day4_DE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_excel('/content/Height_wight_dataset.xlsx')

In [ ]:
# Display first 5 rows
print("First 5 Rows:")
print(df.head())


First 5 Rows:
   Gender   Height_in_inchies  Weight_in_pound 
0    Male                  64               128
1  Female                  66               124
2  Female                  62               136
3    Male                  70               153
4    Male                  68               144


In [ ]:
# Display dataset information
print("\nDataset Info:")
print(df.info())



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Gender             9 non-null      object
 1   Height_in_inchies  9 non-null      int64 
 2   Weight_in_pound    9 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 348.0+ bytes
None


In [ ]:
# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
Gender               0
Height_in_inchies    0
Weight_in_pound      0
dtype: int64


In [ ]:
# Basic statistics
print("\nStatistical Summary:")
print(df.describe())


Statistical Summary:
       Height_in_inchies  Weight_in_pound 
count           9.000000          9.000000
mean           65.777778        134.666667
std             2.538591         13.086252
min            62.000000        116.000000
25%            64.000000        124.000000
50%            66.000000        136.000000
75%            68.000000        144.000000
max            70.000000        153.000000


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
print(df.columns)

# Select columns using position
X = df.iloc[:, [0]]   # First column
y = df.iloc[:, 1]     # Second column


Index(['Gender ', 'Height_in_inchies', 'Weight_in_pound '], dtype='object')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Convert text column into numeric
le = LabelEncoder()

X_train = X_train.apply(le.fit_transform)
X_test = X_test.apply(le.fit_transform)

# Train model
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

print(y_pred)

[64. 68.]


In [ ]:
# Train model
model.fit(X_train, y_train)

# Predict values
y_pred = model.predict(X_test)

# Print prediction
print(y_pred)

[64. 68.]


In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

In [ ]:
print("MAE :", mean_absolute_error(y_test, y_pred))

# Mean Squared Error
print("MSE :", mean_squared_error(y_test, y_pred))

# R2 Score
print("R2 Score :", r2_score(y_test, y_pred))

MAE : 2.0
MSE : 8.0
R2 Score : 0.0


In [ ]:
import joblib

In [ ]:
joblib.dump(model, "linear.pkl")

['linear.pkl']

In [ ]:
a=joblib.load("linear.pkl")

In [ ]:
a.predict([[100]])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([464.])

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, month, to_date, col, round as spark_round

import matplotlib.pyplot as plt
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

# Create SparkSession
spark = SparkSession.builder \
    .appName('Day4_BigData_Sales') \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("Spark version:", spark.version)

print("SparkSession is ACTIVE")

print("Application Name:", spark.sparkContext.appName)

Spark version: 4.0.2
SparkSession is ACTIVE
Application Name: Day4_BigData_Sales


In [ ]:
df_bronze = spark.read \
    .option('header', 'true') \
    .option("inferSchema", "true") \
    .csv('large_sales_data.csv')

print('=== BRONZE LAYER - Raw Data ===')

print(f'Rows : {df_bronze.count()}')

print(f'Columns : {len(df_bronze.columns)}')

print(f'Names : {df_bronze.columns}')

print()

df_bronze.printSchema()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/large_sales_data.csv. SQLSTATE: 42K03

In [ ]:
df_bronze.write \
      .node('overwrite') \
      .parquet('sales_bronze.parquet')

print('Bronze Parquet saved: ales_bronze.parguet')

import os

In [ ]:
df_silver = df_bronze \
   .dropDuplicates() \
   .dropna(subset=['order_id', 'product','revenue'])


In [ ]:
df_silver= df_silver.withColumn(
    'order_date', to_date(col('order_date'), 'yyy-MM-dd')
)

df_silver = df_silver \
    .withColumn('order_year', year(col('order_date'))) \
    .withColumn('order_month', month(col('order_date')))

df_silver =df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue')>40000.'High')
    .when(col('revenue')>20000,'Medium')
    .otherwise('Low')
)

print(f"Silver layer ")

verify by reading it back

print("Silver Parquet saved: sales_silver.parquet")

In [ ]:
import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path)/ 1024
  total =0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      fp = os.path.join(dirpath, f)
      total += os.path.getsize(fp)
  return total/1024

In [ ]:
top_products = df_silver \
 .

 revenue by region